In [2]:
import os
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Plot simulation

In [27]:
num_hosts = 16
nodes = 16 # This is the nodes used for greedy local algorithm, 1 if global
processors_per_host = 36
processors = num_hosts * processors_per_host

resolution = 360  # Resolution of the workload

groupings = {
    "Round Robin": f"test/processor_group/c{resolution}_p{processors}/rr_groups.csv",
    "Greedy Group": f"test/processor_group/c{resolution}_p{processors}/greedy_groups.csv",
    # "MIP Group": f"test/processor_group/c{resolution}_p{processors}/mip_groups.csv",
    "Original Group": None,
}

test_name_base = f"greedy/c{resolution}_p{processors}"

test_names = {}
seeds = {}
pathes = {}
for key, grouping in groupings.items():
    test_name = test_name_base
    seeds[key] = 0
    if nodes > 1:
        test_name += f"_h{nodes}"
        if grouping:
            if grouping.isdigit():
                seed = int(grouping)
                seeds[key] = seed
                test_name += f"_s{seed}"
            else:
                path = Path(grouping)
                pathes[key] = path
                test_name += f"_f{path.stem}"
    test_names[key] = test_name

test_names

{'Round Robin': 'greedy/c360_p576_h16_frr_groups',
 'Greedy Group': 'greedy/c360_p576_h16_fgreedy_groups',
 'Original Group': 'greedy/c360_p576_h16'}

In [28]:
simulation_data = {}
for key, test_name in test_names.items():
    # simulation_data_path = f"test/{test_name}/simulation.csv"
    simulation_data_path = "test/generated_assignments/c360_p576_simulation.csv"
    simulation_data[key] = pd.read_csv(simulation_data_path, index_col=0)
    # Exclude the last four columns and select only numeric columns
    simulation_data[key] = simulation_data[key].iloc[:, :-4].select_dtypes(include=[np.number])

# Print the first item in simulation_data dict
first_key = next(iter(simulation_data))
simulation_data[first_key]

,Processor0,Processor1,Processor2,Processor3,Processor4,Processor5,Processor6,Processor7,Processor8,Processor9,...,Processor566,Processor567,Processor568,Processor569,Processor570,Processor571,Processor572,Processor573,Processor574,Processor575
Interval,,,,,,,,,,,,,,,,,,,,,
0,223135.25,236085.0000,225576.2500,237802.3750,249351.1250,198744.7500,233208.7500,295705.00,280281.50,224695.3750,...,211122.7500,183622.50,239492.50,207959.4375,182036.5625,239652.3750,252228.6250,222635.5000,202188.0000,230498.00
1,187615.50,190032.5000,198494.7500,227969.5625,241646.1875,191535.8125,202958.1875,206854.00,222161.75,200216.0000,...,202921.5625,167642.75,223090.75,199103.6875,177844.8125,221164.5000,235252.0000,195897.5625,189833.4375,187959.75
2,178876.50,181650.7500,195657.0000,220654.6250,240226.1250,186357.6250,191614.3750,194922.00,215631.50,192216.3750,...,196127.1875,163266.00,217007.50,204215.1250,173062.6250,217242.9375,231502.8125,188377.4375,181481.3125,183967.25
3,175842.75,176023.9375,191139.5625,217556.3125,240189.9375,186653.0000,187577.5000,193237.50,211428.75,190005.0000,...,195127.2500,162073.25,212358.25,203799.9375,173885.5625,217316.3125,230205.9375,184842.0625,176178.1875,181145.25
4,175247.25,176887.2500,190265.5000,215664.5000,240542.7500,187489.1875,185274.0625,187980.50,206886.00,186306.0000,...,191923.4375,160995.00,211262.00,206307.8125,174003.1875,214834.1250,230140.6250,182875.3750,173532.8750,180298.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,186742.25,209575.3750,221592.1250,211793.8750,187203.8750,181509.6875,226249.5625,269273.75,220418.75,253609.3750,...,212695.3125,309661.25,183097.50,187724.8750,153639.8750,163168.3125,163245.1875,185613.1875,226996.5625,296732.00
500,180249.00,204152.8125,220375.4375,210304.7500,185172.2500,181791.1875,227126.3125,267976.75,212824.75,246579.9375,...,211899.2500,310623.75,176512.75,183816.6250,151458.3750,162553.7500,163059.7500,186268.1875,226960.5625,294572.50
501,173203.75,200081.0000,216533.2500,208978.4375,186014.5625,180685.7500,227185.2500,268544.00,202825.25,238851.0000,...,211364.3125,310772.75,170043.50,182163.7500,149457.5000,159403.8750,161325.3750,185979.1875,226749.0625,292778.25


In [29]:
# Plot the simulation data as a heatmap,
def plot_simulation_heatmap(data: pd.DataFrame, heatmap_type: str = "processor", path: Optional[str] = None):
    """
    Plot a heatmap of the simulation data.
    Args:
        data (pd.DataFrame): The data to plot.
        heatmap_type (str): Label for the x-axis (e.g., 'processor', 'node').
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(12, 8))
    cax = plt.imshow(data, aspect="auto", cmap="viridis")
    plt.colorbar(cax, label="Value")
    plt.xlabel(heatmap_type.capitalize())
    plt.ylabel("Interval")
    plt.title(f"{heatmap_type.capitalize()} Data Heatmap")
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/{heatmap_type}_heatmap.png")
        plt.savefig(f"{path}/{heatmap_type}_heatmap.eps")
    plt.show()

In [30]:
# # Plot the simulation data heatmap
# plot_simulation_heatmap(
#     simulation_data, heatmap_type="processor", path=f"test/plots/{test_name}"
# )

Generate random grouping or read from input file

In [31]:
def generate_group_from_seed(
    num_hosts: int, processors_per_host: int, seed: int = 0
 ) -> pd.DataFrame:
    """
    Generate processor groups based on a seed value.
    Args:
        seed (int): Seed for random number generator.
        num_hosts (int): Number of hosts.
        processors_per_host (int): Number of processors per host.
    Returns:
        pd.DataFrame: DataFrame with processor groups.
    """
    processors = num_hosts * processors_per_host
    indices = np.arange(processors)
    if seed:
        np.random.seed(seed)
        np.random.shuffle(indices)
    processor_groups = [
        list(indices[i * processors_per_host : (i + 1) * processors_per_host])
        for i in range(num_hosts)
    ]
    return pd.DataFrame(processor_groups)

node_dfs = {}
for key, value in test_names.items():
    if key in pathes:
        node_dfs[key] = pd.read_csv(pathes[key], header=None)
    elif key in seeds:
        node_dfs[key] = generate_group_from_seed(
            num_hosts, processors_per_host, seeds[key]
        )

# Print the first item in node_dfs dict
first_key = next(iter(node_dfs))
node_dfs[first_key]

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,0,16,32,48,64,80,96,112,128,144,...,416,432,448,464,480,496,512,528,544,560
1,1,17,33,49,65,81,97,113,129,145,...,417,433,449,465,481,497,513,529,545,561
2,2,18,34,50,66,82,98,114,130,146,...,418,434,450,466,482,498,514,530,546,562
3,3,19,35,51,67,83,99,115,131,147,...,419,435,451,467,483,499,515,531,547,563
4,4,20,36,52,68,84,100,116,132,148,...,420,436,452,468,484,500,516,532,548,564
5,5,21,37,53,69,85,101,117,133,149,...,421,437,453,469,485,501,517,533,549,565
6,6,22,38,54,70,86,102,118,134,150,...,422,438,454,470,486,502,518,534,550,566
7,7,23,39,55,71,87,103,119,135,151,...,423,439,455,471,487,503,519,535,551,567
8,8,24,40,56,72,88,104,120,136,152,...,424,440,456,472,488,504,520,536,552,568
9,9,25,41,57,73,89,105,121,137,153,...,425,441,457,473,489,505,521,537,553,569


Plot group's total simulated workload

In [32]:
dynamic_summed_dfs = {}

for key, node_df in node_dfs.items():
    dynamic_node_mapping = {}
    for group_id, row in node_df.iterrows():
        for processor_id in row:
            dynamic_node_mapping[f"Processor{processor_id}"] = group_id

    # Rename columns in workload_df using the dynamic node mapping
    dynamic_grouped_df = simulation_data[key].rename(columns=dynamic_node_mapping)
    # Sum workload by group
    dynamic_summed_dfs[key] = dynamic_grouped_df.T.groupby(level=0).sum().T

# Print the first item in dynamic_summed_dfs dict
first_key = next(iter(dynamic_summed_dfs))
dynamic_summed_dfs[first_key]

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
Interval,,,,,,,,,,,,,,,,
0,11244276.50,1.106243e+07,1.177113e+07,1.123412e+07,1.149424e+07,1.294462e+07,1.358834e+07,13090697.00,11278581.50,1.125134e+07,1.177356e+07,1.131649e+07,1.160614e+07,1.277025e+07,1.308709e+07,12491545.00
1,9407379.50,9.908723e+06,1.045407e+07,9.758068e+06,1.030775e+07,1.142009e+07,1.132382e+07,10630148.25,9587530.50,1.009659e+07,1.099837e+07,9.902514e+06,1.040877e+07,1.145539e+07,1.116861e+07,10391153.75
2,9125174.25,9.800259e+06,1.074443e+07,9.866422e+06,1.017810e+07,1.098926e+07,1.073406e+07,9978352.75,9207541.75,1.012265e+07,1.122154e+07,9.970001e+06,1.059781e+07,1.101301e+07,1.069260e+07,10237829.25
3,9023302.25,1.006785e+07,1.103063e+07,9.909115e+06,1.010047e+07,1.058881e+07,1.036330e+07,9904010.25,8855255.75,1.029331e+07,1.139168e+07,1.025640e+07,1.053049e+07,1.031588e+07,1.034163e+07,9881987.75
4,9075461.00,1.066980e+07,1.117624e+07,1.020823e+07,1.008963e+07,1.011712e+07,1.013078e+07,9974326.00,8857435.00,1.047138e+07,1.144474e+07,1.055146e+07,1.008537e+07,9.912681e+06,1.000294e+07,9568536.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,11780822.25,1.008541e+07,8.686447e+06,8.851748e+06,9.998262e+06,1.074550e+07,1.243987e+07,14162802.75,12035891.75,1.072658e+07,8.930992e+06,8.832619e+06,9.615226e+06,1.083409e+07,1.185192e+07,12971285.25
500,11157773.50,1.007202e+07,8.857550e+06,8.893991e+06,9.901011e+06,1.084275e+07,1.241296e+07,13986297.25,11644434.50,1.054275e+07,8.815476e+06,8.756012e+06,9.820228e+06,1.081285e+07,1.218520e+07,12602586.75
501,10687545.50,1.018968e+07,8.952130e+06,8.947094e+06,9.974420e+06,1.112342e+07,1.240089e+07,13178322.75,11156264.50,1.033589e+07,8.860870e+06,8.900456e+06,9.884644e+06,1.077493e+07,1.249620e+07,12198927.25


In [33]:
# Max group workload per interval
max_group_workloads = {}
max_group_workloads_per_processor = {}
for key, dynamic_summed_df in dynamic_summed_dfs.items():
    max_group_workload = dynamic_summed_df.max(axis=1)
    max_group_workloads[key] = max_group_workload
    max_group_workloads_per_processor[key] = max_group_workload / processors_per_host

# Print the first item in max_group_workloads_per_processor dict
first_key = next(iter(max_group_workloads_per_processor))
max_group_workloads_per_processor[first_key]

Interval
0      377453.824653
1      318205.406250
2      311709.335069
3      316435.416667
4      317909.354167
           ...      
499    393411.187500
500    388508.256944
501    366064.520833
502    348484.130208
503    344813.322917
Length: 504, dtype: float64

In [34]:
# Span is the sum of the max group workload across all intervals, divided by the number of processors per host
spans = {}
for key, max_group_workload in max_group_workloads.items():
    spans[key] = max_group_workload.sum() / processors_per_host

spans

{'Round Robin': np.float64(174082373.56770834),
 'Greedy Group': np.float64(174585716.49479166),
 'Original Group': np.float64(253912024.70833334)}

In [35]:
# Calculate the lower bound for all groupings to verify they are the same
lower_bounds = {}
for key, df in simulation_data.items():
    lower_bounds[key] = df.sum().sum() / processors

lower_bounds

{'Round Robin': np.float64(149209568.64583334),
 'Greedy Group': np.float64(149209568.64583334),
 'Original Group': np.float64(149209568.64583334)}

In [36]:
# Plot the span as a bar chart
def plot_span_bar_chart(
    spans: dict, path: Optional[str] = None
):
    """
    Plot a bar chart of the span for each test.
    Args:
        spans (dict): Dictionary with test names as keys and spans as values.
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(10, 6))
    plt.bar(spans.keys(), spans.values(), color='skyblue')
    plt.xlabel('Test Name')
    plt.ylabel('Span')
    plt.title('Span for Each Test')
    plt.xticks(rotation=45)
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/span_bar_chart.png")
        plt.savefig(f"{path}/span_bar_chart.eps")

# Plot the span bar chart
plot_span_bar_chart(
    spans, path=f"test/plots/{test_name_base}"
)

Workload

In [37]:
# Plot the node workload as a bar chart for one interval
def plot_node_workload_bar_chart(data: pd.Series, interval: int, path: Optional[str] = None):
    """
    Plot a bar chart of the node workload for a specific interval.
    Args:
        data (pd.Series): The data to plot.
        interval (int): The interval to plot.
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(12, 6))
    data.plot(kind='bar', label='Node')
    plt.title(f"C180 Node Workload for Interval {interval}")
    plt.xlabel("Node")
    plt.ylabel("Workload")
    # Add a reference line at y = mean workload
    mean_workload = data.mean()
    plt.axhline(mean_workload, color='red', linestyle='--', label='Mean')
    plt.legend()
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/node_workload_interval_{interval}.png")
        plt.savefig(f"{path}/node_workload_interval_{interval}.eps")
    plt.show()

In [38]:
# Plot for both round robin and original groupings
for key in ["Round Robin", "Original Group"]:
    if key in dynamic_summed_dfs:
        # Select the first interval (0) for plotting
        interval_data = dynamic_summed_dfs[key].iloc[0]
        plot_node_workload_bar_chart(
            interval_data, interval=0, path=f"test/plots/{test_names[key]}"
        )
    else:
        print(f"No data available for {key} in dynamic_summed_dfs.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
invalid command name "124198832071616process_stream_events"
    while executing
"124198832071616process_stream_events"
    ("after" script)
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [39]:
# Plot multiple max node workloads as a line chart for comparison
def plot_max_node_workload(series_map: dict, path: Optional[str] = None):
    """
    Plot multiple maximum node workloads as a line chart for comparison.
    Args:
        series_map (dict): Dictionary mapping label (str) to pd.Series (max workload per node).
        path (Optional[str]): If provided, save the plot to this directory.
    """
    plt.figure(figsize=(12, 6))
    for label, data in series_map.items():
        data.plot(kind='line', marker='o', label=label)
    plt.title("Maximum Node Workload Comparison")
    plt.xlabel("Interval")
    plt.ylabel("Max Workload")
    plt.legend()
    if path:
        os.makedirs(path, exist_ok=True)
        plt.savefig(f"{path}/max_node_workload_comparison.png")
        plt.savefig(f"{path}/max_node_workload_comparison.eps")
    plt.show()

In [40]:
# Normalize workload to original group
normalized_max_group_workloads = {}
if "Original Group" in max_group_workloads:
    original_max_group_workload = max_group_workloads["Original Group"]
else:
    raise ValueError("Original Group not found in max_group_workloads")

for key, max_group_workload in max_group_workloads.items():
    normalized_max_group_workloads[key] = max_group_workload / original_max_group_workload

normalized_max_group_workloads["Original Group"]

Interval
0      1.0
1      1.0
2      1.0
3      1.0
4      1.0
      ... 
499    1.0
500    1.0
501    1.0
502    1.0
503    1.0
Length: 504, dtype: float64

In [41]:
plot_max_node_workload(normalized_max_group_workloads, path=f"test/plots/{test_name_base}")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
invalid command name "124198795327488process_stream_events"
    while executing
"124198795327488process_stream_events"
    ("after" script)
